# Exact Rational Coefficients for Small-z Approximation

This notebook derives the **exact rational Taylor series coefficients** for the gravitational wave projection tensor limits.

It uses **SymPy** to:
1. Define the analytic coefficients $c_1 \dots c_5$ using robust Bessel definitions.
2. Compute the symbolic series expansion around $z=0$.
3. Generate valid **Julia code** with exact rational coefficients (e.g., `11//945`).

In [3]:
import sympy as sp
from sympy import symbols, pi, besselj, sqrt, series, simplify, Rational, Poly

# 1. Setup Symbols
z = symbols('z', real=True)

# 2. Robust Spherical Bessel Function
# We rewrite jn(n, z) -> sqrt(pi/(2z)) * J(n+0.5, z) to avoid SymPy 'jn' series issues
def jn(n, z):
    return sqrt(pi / (2 * z)) * besselj(n + Rational(1, 2), z)

# 3. Define the Analytic Formulas (Verified)
# Note: We use exact Rational(1,2) for fractions
c1_expr = 4*pi * ( -Rational(1,2)*jn(0, z) + jn(1, z)/z + jn(2, z)/(2*z**2) )
c2_expr = 4*pi * ( jn(0, z) - 2*jn(1, z)/z + jn(2, z)/(z**2) )
c3_expr = 8*pi * ( jn(2, z) - jn(3, z)/z )
c4_expr = 4*pi * ( -Rational(1,2)*jn(2, z) - Rational(1,2)*jn(3, z)/z )
c5_expr = 2*pi * jn(4, z)

coeffs = [c1_expr, c2_expr, c3_expr, c4_expr, c5_expr]
names = ['c1', 'c2', 'c3', 'c4', 'c5']

## Compute Series Expansions

In [4]:
# Store expansions
expanded_exprs = []

print("Computing Series Expansions (Order 6)...\n")

for name, expr in zip(names, coeffs):
    # Series expansion up to O(z^6)
    # .removeO() drops the +O(z^6) term so we have a pure polynomial
    ser = series(expr, z, 0, 6).removeO()
    expanded_exprs.append(ser)
    
    # Print for verification
    print(f"{name} = {ser}")

Computing Series Expansions (Order 6)...

c1 = -11*pi*z**4/945 + 4*pi*z**2/21 - 8*pi/15
c2 = 23*pi*z**4/945 - 44*pi*z**2/105 + 8*pi/5
c3 = -32*pi*z**4/945 + 16*pi*z**2/35
c4 = 2*pi*z**4/189 - 16*pi*z**2/105
c5 = 2*pi*z**4/945


## Generate Julia Code with Rational Coefficients

The function below extracts the rational factor from each term (removing $\pi$) and formats it as a Julia rational string (e.g., `11//945`).

In [5]:
def to_julia_rational(coeff_val):
    # Divide by pi to get the scalar factor
    val = simplify(coeff_val / pi)
    
    # Convert to string. SymPy prints rationals like "11/945"
    # We want Julia rationals like "11//945"
    val_str = str(val)
    if "/" in val_str:
        val_str = val_str.replace("/", "//")
    return val_str

def format_for_julia(expr):
    poly = Poly(expr, z)
    
    # Get terms sorted by degree
    # poly.terms() returns [ ((degree,), coeff), ... ]
    sorted_terms = sorted(poly.terms(), key=lambda x: x[0][0])
    
    parts = []
    
    for powers, coeff in sorted_terms:
        degree = powers[0] # Extract integer degree
        
        rat_str = to_julia_rational(coeff)
        
        # Handle sign for formatting
        if rat_str.startswith("-"):
            sign = "-"
            rat_str = rat_str[1:] # Strip sign
        else:
            sign = "+"
            
        # Construct term string
        if degree == 0:
            term_code = rat_str
        elif degree == 2:
            term_code = f"{rat_str} * z2"
        elif degree == 4:
            term_code = f"{rat_str} * z4"
        else:
            term_code = f"{rat_str} * z^{degree}"
            
        # Append to list
        if not parts:
            # First term
            if sign == "-":
                parts.append(f"-{term_code}")
            else:
                parts.append(f"{term_code}")
        else:
            parts.append(f" {sign} {term_code}")
            
    return "".join(parts)

print("function small_z_coeffs(z::Real)")
print("    z2 = z * z")
print("    z4 = z2 * z2")
print("    ")

for name, expr in zip(names, expanded_exprs):
    julia_expr = format_for_julia(expr)
    print(f"    {name} = π * ({julia_expr})")

print("    ")
print("    return SVector{5, Float64}(c1, c2, c3, c4, c5)")
print("end")

function small_z_coeffs(z::Real)
    z2 = z * z
    z4 = z2 * z2
    
    c1 = π * (-8//15 + 4//21 * z2 - 11//945 * z4)
    c2 = π * (8//5 - 44//105 * z2 + 23//945 * z4)
    c3 = π * (16//35 * z2 - 32//945 * z4)
    c4 = π * (-16//105 * z2 + 2//189 * z4)
    c5 = π * (2//945 * z4)
    
    return SVector{5, Float64}(c1, c2, c3, c4, c5)
end
